# Качество данных: проверяемые правила


Выразить полноту, уникальность, допустимый диапазон, согласованность и распределение числового поля.

Нужен `SparkSessionContext` и доступ к таблицам `lakehouse.raw.pg__matches`, `lakehouse.raw.pg__match_players` и `lakehouse.raw.mongo__kill_events`. При отсутствии таблиц ячейка не может выполнить проверку.

Ограничения: Перед публикацией набора, сверкой источников и разбором изменения показателей. Порог должен быть частью контракта; одна проверка не доказывает качество всех строк.


## Основные понятия
Проверка качества должна возвращать компактные показатели и явно показывать нарушенные строки.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

## Примеры


**1. Полнота и диапазон**

Одним проходом посчитайте пропуски и значения длительности вне допустимого диапазона.



In [ ]:
with SparkSessionContext('example-quality-completeness') as spark:
    checks = (
        spark
            .table("lakehouse.raw.pg__matches")
            .agg(
                F.count("*").alias("rows"),
                F.sum(F.col("match_id").isNull().cast("int")).alias("missing_ids"),
                F.sum((F.col("duration") <= 0).cast("int")).alias("invalid_duration"),
            ).first().asDict()
    )

    print(checks)

**2. Уникальность ключа**

Найдите повторяющиеся ключи и ограничьте диагностическую выдачу.



In [ ]:
with SparkSessionContext('example-quality-uniqueness') as spark:
    duplicates = (
        spark
            .table("lakehouse.raw.pg__match_players")
            .groupBy("match_id", "player_slot").count()
            .filter(F.col("count") > 1)
    
    )

    print("Повторяющихся ключей:", duplicates.count())
    duplicates.show(20, truncate=False)

**3. Допустимые значения**

Проверьте, что признак победы не содержит неожиданных значений.



In [ ]:
with SparkSessionContext('example-quality-domain') as spark:
    values = (
        spark
            .table("lakehouse.raw.pg__matches")
            .groupBy("radiant_win").count()
            .orderBy("radiant_win")
    )
    values.show(truncate=False)

**4. Согласованность наборов**

Найдите события, для которых нет соответствующего матча.



In [ ]:
with SparkSessionContext('example-quality-reconciliation') as spark:
    matches = spark.table("lakehouse.raw.pg__matches").select("match_id").distinct()
    events = spark.table("lakehouse.raw.mongo__kill_events").select("match_id").distinct()
    events.join(matches, "match_id", "left_anti").limit(20).show(truncate=False)

**5. Распределение**

Получите минимальное, среднее, медианное и максимальное значение.



In [ ]:
with SparkSessionContext('example-quality-distribution') as spark:
    (
        spark
        .table("lakehouse.raw.pg__matches")
        .select("duration")
        .filter("duration > 0")
        .agg(
            F.min("duration").alias("minimum"), 
            F.avg("duration").alias("average"),
            F.expr("percentile_approx(duration, 0.5)").alias("median"), 
            F.max("duration").alias("maximum")
        )
        .show(truncate=False)
    )
